In [1]:
# Biblioteker

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
# Indlæs data
gdp = pd.read_csv("GDP.csv", skiprows=4) #GDP per capita
aging = pd.read_csv("age-dependency-ratio-old.csv") #% of working-age population
pwt = pd.read_excel("Human-Productivity.xlsx", sheet_name="Data") #data from penworld

In [3]:
# Tjek de har læst korrekt

print(gdp.head())
print(gdp.columns[:10])

print(aging.head())
print(aging.columns)

print(pwt.head())
print(pwt.columns[:15])

                  Country Name Country Code  \
0                        Aruba          ABW   
1  Africa Eastern and Southern          AFE   
2                  Afghanistan          AFG   
3   Africa Western and Central          AFW   
4                       Angola          AGO   

                                  Indicator Name     Indicator Code  1960  \
0  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
1  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
2  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
3  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
4  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   

   1961  1962  1963  1964  1965  ...          2017          2018  \
0   NaN   NaN   NaN   NaN   NaN  ...  37524.914920  39287.059713   
1   NaN   NaN   NaN   NaN   NaN  ...   3837.726375   3723.216423   
2   NaN   NaN   NaN   NaN   NaN  ...   2335.795862

In [4]:
gdp = gdp.rename(columns={"Country Name": "country"})

gdp = gdp.melt(
    id_vars=["country", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="year",
    value_name="gdp"
)

In [5]:
gdp = gdp[["country", "year", "gdp"]]

In [6]:
gdp["year"] = pd.to_numeric(gdp["year"], errors="coerce")
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")

In [7]:
aging = aging.rename(columns={
    "Entity": "country",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "aging"
})

In [8]:
aging = aging[["country", "year", "aging"]]

In [9]:
aging["year"] = pd.to_numeric(aging["year"], errors="coerce")
aging["aging"] = pd.to_numeric(aging["aging"], errors="coerce")

In [10]:
pwt = pwt[["country", "year", "rkna", "emp", "rtfpna"]].copy()

In [11]:
pwt = pwt.rename(columns={
    "rkna": "K",
    "emp": "L",
    "rtfpna": "A"
})

In [12]:
pwt["year"] = pd.to_numeric(pwt["year"], errors="coerce")
pwt["K"] = pd.to_numeric(pwt["K"], errors="coerce")
pwt["L"] = pd.to_numeric(pwt["L"], errors="coerce")
pwt["A"] = pd.to_numeric(pwt["A"], errors="coerce")

In [13]:
# Merge GDP og PWT på country og year 

df = gdp.merge(pwt, on=["country", "year"], how="inner")

In [14]:
# Merge aging på 

df = df.merge(aging, on=["country", "year"], how="inner")

In [15]:
# Fjern manglende værdier

df = df.dropna()

In [16]:
# gør year til int
df["year"] = df["year"].astype(int)

# Tidsperiode 1990-2020
df = df[(df["year"] >= 1990) & (df["year"] <= 2020)]

In [17]:
print(df.head())
print(df.shape)
print(df.describe())

        country  year           gdp         K          L         A      aging
4561     Angola  1990   3704.863177  0.275776   5.741761  1.219142   4.293690
4562    Albania  1990   2655.673133  0.318261   1.238229  0.477820   8.680737
4564  Argentina  1990   7158.308492  0.446382  11.811479  0.938284  14.126203
4567  Australia  1990  17385.498542  0.336556   8.066450  0.812855  16.522020
4568    Austria  1990  19395.859899  0.425653   3.540237  0.934566  22.141403
(3178, 7)
              year            gdp            K            L            A  \
count  3178.000000    3178.000000  3178.000000  3178.000000  3178.000000   
mean   2005.195091   18195.463495     0.603246    21.817121     1.008825   
std       8.859748   19768.329777     0.251389    85.246082     0.263304   
min    1990.000000     291.989333     0.035137     0.094950     0.171880   
25%    1998.000000    4400.798046     0.410822     1.413827     0.899355   
50%    2005.000000   10947.132352     0.595277     3.730232     0.

In [18]:
# Log variabler

df["log_gdp"] = np.log(df["gdp"])
df["log_K"] = np.log(df["K"])
df["log_L"] = np.log(df["L"])
df["log_A"] = np.log(df["A"])

In [19]:
# log variabler
df["log_gdp"] = np.log(df["gdp"])
df["log_K"] = np.log(df["K"])
df["log_L"] = np.log(df["L"])
df["log_A"] = np.log(df["A"])

# forklarende variabler
X = df[["log_K", "log_L", "log_A", "aging"]]
X = sm.add_constant(X)

# afhængig variabel
y = df["log_gdp"]

# regression
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                log_gdp   R-squared:                       0.529
Model:                            OLS   Adj. R-squared:                  0.528
Method:                 Least Squares   F-statistic:                     889.6
Date:                Fri, 03 Apr 2026   Prob (F-statistic):               0.00
Time:                        10:24:27   Log-Likelihood:                -3920.0
No. Observations:                3178   AIC:                             7850.
Df Residuals:                    3173   BIC:                             7880.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          8.3414      0.040    207.422      0.0